# Task 3

In [ ]:
import HMM_inference
import matplotlib.pyplot as plt
import numpy as np
from HMM_models import RampModelHMM, StepModelHMM

In [ ]:
plt.rcParams['axes.titlesize'] = 18  # Set default title font size
plt.rcParams['figure.titlesize'] = 22
plt.rcParams['font.family'] = ['cmr10', 'Times New Roman', 'STIXGeneral']
plt.rcParams['xtick.labelsize'] = 18  # Font size of x-axis tick labels
plt.rcParams['ytick.labelsize'] = 18  # Font size of y-axis tick labels

## Task 3.1.1.

Visualising the approximate posterior on the grid, repeated for different true parameter values and for different numbers of trials for each model.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# Example varying parameter - here N changes
N_values = [1, 20, 100, 400]

for ax, N in zip(axs.flat, N_values):
    _ = HMM_inference.ramp_inference_scan(N=N,
                                        prior_type='uniform', 
                                        ax=ax,
                                        plot=True)
    ax.set_title(f'N={N}')


plt.suptitle('Ramp Model Posterior Scan for Different Sample Sizes \n (Flat Prior)', fontsize=22)
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)


N_values = [1, 20, 100, 400]
for ax, N in zip(axs.flat, N_values):
    _ = HMM_inference.step_inference_scan(true_m=200,
                                                    true_r=4,
                                                    ax=ax, 
                                                    prior_type='uniform',
                                                    N=N,
                                                    plot=True)
    ax.set_title(f'Sample Size={N}')

plt.suptitle('Step Model Posterior Scan for Different Sample Sizes \n (Flat Prior)', fontsize=22)
plt.show()

In [ ]:
fixed_N = 20
beta_true_values = [0.5, 1, 1.5]
sigma_true_values = [0.5, 1, 3]

fig, axs = plt.subplots(3, 3, figsize=(15, 12), constrained_layout=True)

for i, true_beta in enumerate(beta_true_values):
    for j, true_sigma in enumerate(sigma_true_values):
        ax = axs[i, j]
        _ = HMM_inference.ramp_inference_scan(
            true_beta=true_beta,
            true_sigma=true_sigma,
            prior_type='uniform',
            N=fixed_N,
            ax=ax,
            plot=True
        )
        ax.set_title(rf'Model: $\beta$={true_beta}, $\sigma$={true_sigma}')

plt.suptitle('Ramp Model Posterior for a range of Parameters', fontsize=22)
plt.show()


In [ ]:
fixed_N = 20
T = 500
m_true_values = [0.35*T, 0.5*T, 0.65*T]
r_true_values = [1, 3, 5]

fig, axs = plt.subplots(3, 3, figsize=(15, 12), constrained_layout=True)

for i, m in enumerate(m_true_values):
    for j, r in enumerate(r_true_values):
        ax = axs[i, j]
        _ = HMM_inference.step_inference_scan(
            true_m=m,
            true_r=r,
            N=fixed_N,
            prior_type='uniform',
            ax=ax
        )
        ax.set_title(rf'Model: m={m}, r={r}')

plt.suptitle('Step Model Posterior for a range of Parameters \n (Flat Prior)')
plt.show()


## Task 3.1.2.

Evaluation of posterior expectations of parameters, measuring error as difference between the true and predicted parameters and plotting how this error changes with number of trials and choices of true parameters.

## Task 3.2.

Use a truncated Gaussian prior centred on the midpoint of the investigated range and attempt to classify using Bayes Factor (Marginal Likelihood Ratio)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# vary the size of the gaussian prior
f = [0.1, 0.5, 2, 3]

for ax, sd in zip(axs.flat, f):
    _ = HMM_inference.ramp_inference_scan(ax=ax,
                                    N=20, 
                                    prior_type='gaussian',
                                    prior_sd_fraction=sd)
    ax.set_title(f'Gaussian Width = {sd}')

plt.suptitle('Ramp Model Posterior, varying width of central Gaussian prior')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# vary the size of the gaussian prior
f = [0.1, 0.5, 2, 3]

for ax, sd in zip(axs.flat, f):
    _ = HMM_inference.step_inference_scan(true_m=250,
                                                    ax=ax, 
                                                    prior_type='gaussian',
                                                    prior_sd_fraction=sd)
    ax.set_title(f'Gaussian Width = {sd}')

plt.suptitle('Step Model Posterior, varying width of central Gaussian prior')
plt.show()

Now we use the outputs of the functions to compare the Marginal Likelihoods and compare the Bayes' Factors

In [ ]:
# simulate some spike train ensembles from each model

s0=0.2
fixed_rh = 50
T = 500
dt = 1/T
K=100
true_beta = 1
true_sigma = 1
n_trials = 50

ramp_model = RampModelHMM(K=K, beta=true_beta, sigma=true_sigma, dt=dt)
ramp_spike_trains = np.array([
    ramp_model.simulate_spikes(n_steps=T, initial_state=s0, R_h=fixed_rh, dt=dt)[2]
    for _ in range(n_trials)
])

step_model = StepModelHMM()
step_spike_trains = np.array([step_model.simulate_spikes(n_steps=T)[2]
                             for _ in range(n_trials)
])

In [ ]:
# run inference from both models on the ramp model spike trains:

_, _, _, ramp_ml, _ = HMM_inference.ramp_inference_scan(spktrn_arg=ramp_spike_trains, plot=False)
_, _, _, step_ml, _ = HMM_inference.step_inference_scan(spktrn_arg=ramp_spike_trains, plot=False)

In [ ]:
print(ramp_ml)
print(step_ml)

Profiling cell to check where the code is taking longest to run:

In [ ]:
import cProfile
import pstats
from io import StringIO

pr = cProfile.Profile()
pr.enable()

# Run the grid inference function
HMM_inference.ramp_inference_scan(
    true_beta=1,
    true_sigma=0.3,
    N=50,
    M=40,
)
pr.disable()

# Process stats
s = StringIO()
ps = pstats.Stats(pr, stream=s).strip_dirs().sort_stats("tottime")
ps.print_stats(20)  # Show top 20

# Display
print(s.getvalue())
